# EDA train

In [3]:
import pandas as pd

# 1. Carregar o dataset de forma robusta
# O engine='python' é mais lento, mas lida melhor com erros de aspas/formatação
try:
    df = pd.read_csv('../data/train.csv')
    print("✅ Dataset carregado com sucesso!\n")
except Exception as e:
    print(f"❌ Erro ao carregar o arquivo: {e}")

# 2. Printar a quantidade de nulos em cada coluna
print("Quantidade de valores nulos por coluna:")
print(df.isnull().sum())

# 3. Mostrar as dimensões e o cabeçalho para conferência
print(f"\nDimensões do dataset: {df.shape}")
print("\nPrimeiras linhas:")
display(df.head())

✅ Dataset carregado com sucesso!

Quantidade de valores nulos por coluna:
id         0
title      0
text       0
subject    0
date       0
label      0
dtype: int64

Dimensões do dataset: (22844, 6)

Primeiras linhas:


,id,title,text,subject,date,label
0,13355,"Exclusive: Pentagon, Lockheed near deal on $9 ...",WASHINGTON (Reuters) - The U.S. Department of ...,politicsNews,"January 19, 2017",0
1,2113,“HILL”ARIOUS…MUST SEE! IOWA PARADE GOERS Treat...,HILL larious! If this was a Donald Trump pi a...,left-news,"Aug 3, 2016",1
2,27667,Philippine leader says 'no way' he'll do deal ...,MANILA (Reuters) - Philippine President Rodrig...,worldnews,"September 9, 2017",0
3,15368,Biden asks U.S. Congress to allow unencumbered...,WASHINGTON (Reuters) - Vice President Joe Bide...,politicsNews,"September 8, 2016",0
4,6934,Trump Claims ‘Any Negative Polls’ Are ‘FAKE N...,Donald Trump kicked off his Monday morning by ...,News,"February 6, 2017",1


In [5]:
import pandas as pd
import re

def run_data_audit(df):
    print("" + "="*50)
    print("      RELATÓRIO DE AUDITORIA DE DADOS (NLP)")
    print("="*50 + "\n")

    # --- 2. INTEGRIDADE E DUPLICATAS ---
    dups = df.duplicated(subset=['title', 'text']).sum()
    print(f"[!] Duplicatas (Título + Texto): {dups}")
    print("    Nota: Devem ser removidas para evitar que o modelo decore respostas.\n")

    # --- 3. QUALIDADE DO TEXTO (CURTOS/VAZIOS) ---
    empty_title = (df['title'].str.strip() == "").sum()
    empty_text = (df['text'].str.strip() == "").sum()
    short_text = (df['text'].str.len() < 100).sum()

    print(f"[!] Títulos vazios: {empty_title}")
    print(f"[!] Textos vazios/apenas espaços: {empty_text}")
    print(f"[!] Textos muito curtos (< 100 caracteres): {short_text}")
    print("    Nota: Linhas sem conteúdo textual útil são ruído para o BERT.\n")

    # --- 4. DATA LEAKAGE (Reuters) ---
    reuters_count = df['text'].str.contains(r'\(Reuters\)', case=False, na=False).sum()
    print(f"[!] Menções à '(Reuters)': {reuters_count}")
    print(f"    Proporção: {(reuters_count/len(df)*100):.2f}% do dataset.")
    print("    Nota: Isso é 'leakage'. O modelo pode ficar preguiçoso e focar na fonte, não no conteúdo.\n")

    # --- 5. RUÍDOS TÉCNICOS (LINKS/HTML) ---
    links_count = df['text'].str.contains(r'http[s]?://', na=False).sum()
    print(f"[!] Linhas com URLs/Links: {links_count}")
    print("    Nota: URLs raramente ajudam na classificação de desinformação.\n")

    # --- 6. EQUILÍBRIO DE CLASSES ---
    # O F1-Score é afetado se houver muita discrepância entre as classes
    balance = df['label'].value_counts()
    print("[!] Distribuição de Labels:")
    for label, count in balance.items():
        percent = (count / len(df)) * 100
        status = "Verdadeiro (Real)" if label == 0 else "Desinformação (Fake)"
        print(f"    - Label {label} ({status}): {count} ({percent:.2f}%)")

    if len(balance) > 1:
        ratio = max(balance) / min(balance)
        print(f"    Desbalanceamento: 1 para {ratio:.2f}")

    print("\n" + "="*50)

# Executar a auditoria
run_data_audit(df)

      RELATÓRIO DE AUDITORIA DE DADOS (NLP)

[!] Duplicatas (Título + Texto): 501
    Nota: Devem ser removidas para evitar que o modelo decore respostas.

[!] Títulos vazios: 0
[!] Textos vazios/apenas espaços: 154
[!] Textos muito curtos (< 100 caracteres): 257
    Nota: Linhas sem conteúdo textual útil são ruído para o BERT.

[!] Menções à '(Reuters)': 16997
    Proporção: 74.40% do dataset.
    Nota: Isso é 'leakage'. O modelo pode ficar preguiçoso e focar na fonte, não no conteúdo.

[!] Linhas com URLs/Links: 811
    Nota: URLs raramente ajudam na classificação de desinformação.

[!] Distribuição de Labels:
    - Label 0 (Verdadeiro (Real)): 17133 (75.00%)
    - Label 1 (Desinformação (Fake)): 5711 (25.00%)
    Desbalanceamento: 1 para 3.00



In [6]:
# Filtrar apenas as linhas que contêm "(Reuters)"
reuters_df = df[df['text'].str.contains(r'\(Reuters\)', case=False, na=False)]

# Calcular as porcentagens por label
counts = reuters_df['label'].value_counts(normalize=True) * 100

print(f"Total de notícias com '(Reuters)': {len(reuters_df)}")
print(f"Porcentagem que são Label 0 (Verdade): {counts.get(0, 0):.2f}%")
print(f"Porcentagem que são Label 1 (Fake): {counts.get(1, 0):.2f}%")

Total de notícias com '(Reuters)': 16997
Porcentagem que são Label 0 (Verdade): 99.99%
Porcentagem que são Label 1 (Fake): 0.01%


In [7]:
# Agrupar por subject e label
counts = df.groupby(['subject', 'label']).size().unstack(fill_value=0)

# Calcular as porcentagens
percentages = counts.divide(counts.sum(axis=1), axis=0) * 100

print("--- Análise de Subject vs Label ---")
for subject in percentages.index:
    p0 = percentages.loc[subject, 0]
    p1 = percentages.loc[subject, 1]
    total = counts.loc[subject].sum()
    print(f"Subject: {subject: <20} | Total: {total: >5} | Label 0: {p0: >6.2f}% | Label 1: {p1: >6.2f}%")

--- Análise de Subject vs Label ---
Subject: Government News      | Total:   403 | Label 0:   0.00% | Label 1: 100.00%
Subject: Middle-east          | Total:   190 | Label 0:   0.00% | Label 1: 100.00%
Subject: News                 | Total:  2206 | Label 0:   0.00% | Label 1: 100.00%
Subject: US_News              | Total:   197 | Label 0:   0.00% | Label 1: 100.00%
Subject: left-news            | Total:  1071 | Label 0:   0.00% | Label 1: 100.00%
Subject: politics             | Total:  1644 | Label 0:   0.00% | Label 1: 100.00%
Subject: politicsNews         | Total:  9028 | Label 0: 100.00% | Label 1:   0.00%
Subject: worldnews            | Total:  8105 | Label 0: 100.00% | Label 1:   0.00%


# EDA test

In [8]:
import pandas as pd

# 1. Carregar o dataset de forma robusta
# O engine='python' é mais lento, mas lida melhor com erros de aspas/formatação
try:
    df_test = pd.read_csv('../data/test.csv')
    print("✅ Dataset carregado com sucesso!\n")
except Exception as e:
    print(f"❌ Erro ao carregar o arquivo: {e}")

# 2. Printar a quantidade de nulos em cada coluna
print("Quantidade de valores nulos por coluna:")
print(df_test.isnull().sum())

# 3. Mostrar as dimensões e o cabeçalho para conferência
print(f"\nDimensões do dataset: {df_test.shape}")
print("\nPrimeiras linhas:")
display(df_test.head())

✅ Dataset carregado com sucesso!

Quantidade de valores nulos por coluna:
id         0
title      0
text       0
subject    0
date       0
dtype: int64

Dimensões do dataset: (5712, 5)

Primeiras linhas:


,id,title,text,subject,date
0,5398,Obama’s “CLOCK BOY” Comes Back To Texas…After ...,After 9 Months Of Hard-Core Islam Muslim Cloc...,politics,"Jun 27, 2016"
1,5503,LGBT VOLUNTEERS Aren’t Waiting To Be Thrown Of...,A group of volunteer soldiers announced this w...,politics,"Jul 25, 2017"
2,23151,Colombia authorizes air raids against dissiden...,BOGOTA (Reuters) - Colombia s armed forces hav...,worldnews,"October 31, 2017"
3,12669,"Timeline: Trump questions then honors ""one Chi...",(Reuters) - U.S. President Donald Trump agreed...,politicsNews,"February 10, 2017"
4,27864,Three policemen killed in Peru in drug-traffic...,LIMA (Reuters) - Three Peruvian policemen were...,worldnews,"September 7, 2017"


In [9]:
import pandas as pd
import re

def run_data_audit(df):
    print("" + "="*50)
    print("      RELATÓRIO DE AUDITORIA DE DADOS (NLP)")
    print("="*50 + "\n")

    # --- 2. INTEGRIDADE E DUPLICATAS ---
    dups = df.duplicated(subset=['title', 'text']).sum()
    print(f"[!] Duplicatas (Título + Texto): {dups}")
    print("    Nota: Devem ser removidas para evitar que o modelo decore respostas.\n")

    # --- 3. QUALIDADE DO TEXTO (CURTOS/VAZIOS) ---
    empty_title = (df['title'].str.strip() == "").sum()
    empty_text = (df['text'].str.strip() == "").sum()
    short_text = (df['text'].str.len() < 100).sum()

    print(f"[!] Títulos vazios: {empty_title}")
    print(f"[!] Textos vazios/apenas espaços: {empty_text}")
    print(f"[!] Textos muito curtos (< 100 caracteres): {short_text}")
    print("    Nota: Linhas sem conteúdo textual útil são ruído para o BERT.\n")

    # --- 4. DATA LEAKAGE (Reuters) ---
    reuters_count = df['text'].str.contains(r'\(Reuters\)', case=False, na=False).sum()
    print(f"[!] Menções à '(Reuters)': {reuters_count}")
    print(f"    Proporção: {(reuters_count/len(df)*100):.2f}% do dataset.")
    print("    Nota: Isso é 'leakage'. O modelo pode ficar preguiçoso e focar na fonte, não no conteúdo.\n")

    # --- 5. RUÍDOS TÉCNICOS (LINKS/HTML) ---
    links_count = df['text'].str.contains(r'http[s]?://', na=False).sum()
    print(f"[!] Linhas com URLs/Links: {links_count}")
    print("    Nota: URLs raramente ajudam na classificação de desinformação.\n")


# Executar a auditoria
run_data_audit(df_test)

      RELATÓRIO DE AUDITORIA DE DADOS (NLP)

[!] Duplicatas (Título + Texto): 29
    Nota: Devem ser removidas para evitar que o modelo decore respostas.

[!] Títulos vazios: 0
[!] Textos vazios/apenas espaços: 37
[!] Textos muito curtos (< 100 caracteres): 60
    Nota: Linhas sem conteúdo textual útil são ruído para o BERT.

[!] Menções à '(Reuters)': 4252
    Proporção: 74.44% do dataset.
    Nota: Isso é 'leakage'. O modelo pode ficar preguiçoso e focar na fonte, não no conteúdo.

[!] Linhas com URLs/Links: 202
    Nota: URLs raramente ajudam na classificação de desinformação.



# Main

In [11]:
# 1. Instalação com Upgrade e correção de conflito do pyarrow
# Adicionamos 'pyarrow' explicitamente para resolver o ValueError de incompatibilidade binária
!pip install -q -U pyarrow transformers datasets accelerate evaluate

# IMPORTANTE: Se o erro persistir após rodar esta célula pela primeira vez,
# vá em "Ambiente de Execução" -> "Reiniciar sessão" e rode tudo de novo.

import os
import pandas as pd
import numpy as np
import torch
import random
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate

# --- REPRODUTIBILIDADE (Exigência do Desafio) ---
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# Configurações de determinismo para o PyTorch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Definir dispositivo (GPU ou CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Ambiente configurado. Dispositivo: {device}")

✅ Ambiente configurado. Dispositivo: cuda


In [12]:
import pandas as pd
import re

def clean_content(text):
    """
    Limpeza fundamental para evitar Data Leakage.
    Remove apenas a marca da Reuters e URLs, mantendo Maiúsculas e Pontuação.
    """
    text = str(text)
    # 1. Remover Menção à Reuters (Leakage estatístico de 99.9%)
    text = re.sub(r'^.*?\(Reuters\)\s*[-—]\s*', '', text, flags=re.IGNORECASE)

    # 2. Remover URLs (Não agregam valor semântico)
    text = re.sub(r'http\S+', '', text)

    # 3. Normalizar espaços e quebras de linha
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def prepare_train_data(path):
    """
    Processa o treino: remove duplicatas e prepara Título e Texto como colunas distintas.
    """
    # Carregamento robusto para evitar erros de aspas
    df = pd.read_csv(path, engine='python', on_bad_lines='skip')

    # A. Remover Duplicatas (Essencial para integridade científica)
    df = df.drop_duplicates(subset=['title', 'text'])

    # B. Validar Labels
    df = df[df['label'].astype(str).isin(['0', '1', '0.0', '1.0'])]
    df['label'] = df['label'].astype(int)

    # C. Limpeza de Conteúdo (Mantendo Upper Case)
    # Título e Texto permanecem em colunas separadas para o Tokenizer
    df['title_clean'] = df['title'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())
    df['text_clean'] = df['text'].apply(clean_content)

    # D. Filtro de Ruído: Garante que haja conteúdo mínimo para o modelo ler
    df = df[(df['title_clean'].str.len() > 5) & (df['text_clean'].str.len() > 20)]

    print(f"✅ Treino preparado: {df.shape[0]} amostras após limpeza.")
    return df[['id', 'title_clean', 'text_clean', 'label']]

def prepare_test_data(path):
    """
    Processa o teste: mantém a quantidade original de linhas e limpa o texto.
    """
    df = pd.read_csv(path, engine='python', on_bad_lines='skip')

    # A. Limpeza de Conteúdo (Idêntica ao treino para consistência)
    df['title_clean'] = df['title'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())
    df['text_clean'] = df['text'].apply(clean_content)

    # No teste, não removemos linhas, apenas garantimos que não há strings vazias
    # Se o texto estiver vazio após a limpeza, o Título ainda servirá de base

    print(f"✅ Teste preparado: {df.shape[0]} amostras.")
    return df[['id', 'title_clean', 'text_clean']]

In [13]:
def prepare_tokenized_datasets(model_name, df_train, df_test):
    """
    Recebe o nome do modelo e os dataframes limpos.
    Retorna os datasets tokenizados prontos para o modelo.
    """
    print(f"📥 Carregando Tokenizer: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # 1. Converter para formato Dataset
    ds_train = Dataset.from_pandas(df_train)
    ds_test = Dataset.from_pandas(df_test)

    # 2. Função interna de tokenização
    def tokenize_function(batch):
        return tokenizer(
            batch["title_clean"],
            batch["text_clean"],
            padding="max_length",
            truncation=True,
            max_length=256
        )

    # 3. Mapear tokenização
    print("✂️ Tokenizando datasets...")
    tok_train = ds_train.map(tokenize_function, batched=True, desc="Treino")
    tok_test = ds_test.map(tokenize_function, batched=True, desc="Teste")

    # 4. Limpeza de colunas DINÂMICA
    # Lista de colunas que queremos remover se existirem
    cols_to_remove_base = ['title_clean', 'text_clean', '__index_level_0__']

    # Filtramos para remover apenas as que de fato estão no dataset
    actual_cols_train = [c for c in cols_to_remove_base + ['id'] if c in tok_train.column_names]
    actual_cols_test = [c for c in cols_to_remove_base if c in tok_test.column_names]

    tok_train = tok_train.remove_columns(actual_cols_train)
    tok_test = tok_test.remove_columns(actual_cols_test)

    print(f"✅ Tokenização concluída para {model_name}")
    print(f"Colunas finais no treino: {tok_train.column_names}")

    return tok_train, tok_test, tokenizer

In [14]:
# --- CÉLULA 4: FUNÇÃO DE TREINAMENTO ---

def train_model(model_name, tokenized_dataset, tokenizer_obj, epochs=3):
    """
    Realiza o fine-tuning de um modelo específico.
    Retorna o trainer treinado e as métricas de validação.
    """
    print(f"\n⚙️ Configurando treinamento para: {model_name}")

    # 1. Divisão interna para Validação (15%)
    # Usamos o dataset tokenizado que veio da função anterior
    split_ds = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
    train_subset = split_ds["train"]
    eval_subset = split_ds["test"]

    # 2. Carregar o Modelo
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # 3. Métrica F1
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return f1_metric.compute(predictions=predictions, references=labels)

    # 4. Configurar Hiperparâmetros
    # O output_dir é dinâmico para não sobrescrever arquivos de modelos diferentes
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.split('/')[-1]}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        seed=42,
        data_seed=42,
        report_to="none"
    )

    # 5. Inicializar o Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=eval_subset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer_obj),
        compute_metrics=compute_metrics,
    )

    # 6. Treinar
    print(f"🚀 Iniciando treino de {model_name}...")
    trainer.train()

    # Avaliação final para retornar métricas limpas
    metrics = trainer.evaluate()
    print(f"✅ Treino concluído! F1-Score Final: {metrics['eval_f1']:.4f}")

    return trainer, metrics

In [15]:
# --- CÉLULA 5: FUNÇÃO DE PREDIÇÃO E GERAÇÃO DE SUBMISSÃO ---

def generate_submission(trainer_obj, tokenized_test_ds, original_test_df, model_name):
    """
    Usa um trainer treinado para prever o conjunto de teste e gera o CSV de submissão.
    """
    print(f"🔍 Iniciando predições com o modelo: {model_name}...")

    # 1. Gerar predições
    # O trainer utiliza o melhor checkpoint salvo (load_best_model_at_end)
    test_predictions = trainer_obj.predict(tokenized_test_ds)

    # 2. Converter logits para classes (0 ou 1)
    predictions = np.argmax(test_predictions.predictions, axis=-1)

    # 3. Criar o DataFrame de submissão
    # Garantimos o uso do 'id' original para integridade do desafio
    submission_df = pd.DataFrame({
        'id': original_test_df['id'],
        'label': predictions
    })

    # 4. Salvar o arquivo CSV
    # Nomeamos o arquivo com base no modelo para evitar confusão
    filename = f'submission_{model_name.split("/")[-1]}.csv'
    submission_df.to_csv(filename, index=False)

    print(f"\n✅ Ficheiro '{filename}' gerado com sucesso!")
    print(f"📊 Total de linhas: {len(submission_df)}")
    print(f"📈 Distribuição das classes:\n{submission_df['label'].value_counts()}")

    # Tentativa de download automático no Colab
    try:
        from google.colab import files
        files.download(filename)
    except Exception as e:
        print(f"\n[Nota] Download automático falhou. Podes baixar '{filename}' na aba de arquivos.")

    return submission_df

In [16]:
# --- EXECUÇÃO DO PREPROCESSAMENTO ---

train_final = prepare_train_data('train.csv')
test_final = prepare_test_data('test.csv')

# Exemplo de como os dados serão alimentados no modelo
print("\n--- Estrutura Final para o Tokenizer ---")
print(f"Colunas do Treino: {train_final.columns.tolist()}")
print(f"Exemplo Título: {train_final['title_clean'].iloc[0]}")
print(f"Exemplo Texto: {train_final['text_clean'].iloc[0][:150]}...")

✅ Treino preparado: 22169 amostras após limpeza.
✅ Teste preparado: 5712 amostras.

--- Estrutura Final para o Tokenizer ---
Colunas do Treino: ['id', 'title_clean', 'text_clean', 'label']
Exemplo Título: Exclusive: Pentagon, Lockheed near deal on $9 billion F-35 contract - sources
Exemplo Texto: The U.S. Department of Defense and Lockheed Martin Corp (LMT.N) are close to deal for a contract worth almost $9 billion as negotiations are poised to...


In [ ]:
# --- CÉLULA: PIPELINE DE EXPERIMENTAÇÃO COMPLETO ---

def run_full_experiment(model_name, df_train, df_test, epochs=3):
    """
    Executa o fluxo fim-a-fim: Tokenização -> Treino -> Submissão.
    """
    print(f"\n{'='*60}")
    print(f"🏁 INICIANDO EXPERIMENTO: {model_name}")
    print(f"{'='*60}")

    # 1. Fase de Tokenização
    # (Usa a função que criamos na Célula 3)
    tok_train, tok_test, tokenizer_obj = prepare_tokenized_datasets(
        model_name, df_train, df_test
    )

    # 2. Fase de Treinamento
    # (Usa a função que criamos na Célula 4)
    trainer_obj, metrics = train_model(
        model_name, tok_train, tokenizer_obj, epochs=epochs
    )

    # 3. Fase de Submissão
    # (Usa a função que criamos na Célula 5)
    submission_df = generate_submission(
        trainer_obj, tok_test, df_test, model_name
    )

    print(f"\n✨ Experimento com {model_name} finalizado com sucesso!")
    return {
        'model': model_name,
        'f1': metrics['eval_f1'],
        'trainer': trainer_obj
    }

# --- BLOCO DE EXECUÇÃO FINAL (Onde o usuário escolhe) ---

# O usuário só precisa alterar esta lista para testar quantos modelos quiser
modelos_para_testar = ["distilroberta-base", "roberta-base"]

# Dicionário para guardar os resultados de cada modelo para comparação posterior
historico_experimentos = []

for checkpoint in modelos_para_testar:
    resultado = run_full_experiment(checkpoint, train_final, test_final)
    historico_experimentos.append(resultado)

    # Limpeza de memória para o próximo modelo (importante na T4!)
    torch.cuda.empty_cache()
    import gc
    gc.collect()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Obter as predições do conjunto de validação (dados não observados)
val_predictions = trainer.predict(eval_subset)
val_preds = np.argmax(val_predictions.predictions, axis=-1)
val_labels = eval_subset["label"]

# 2. Relatório de Classificação
print("📊 Relatório de Generalização (Conjunto de Validação):")
print(classification_report(val_labels, val_preds, target_names=['Real (0)', 'Fake (1)']))

# 3. Matriz de Confusão para provar a estabilidade
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Estabilidade em Dados Não Observados')
plt.show()

# Stress test

In [ ]:
# Versão rápida para o Teste de Estresse
# Carrega a métrica F1 uma única vez
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Função global para calcular o F1-Score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels)


def run_stress_test(model_name, df_train, df_test, lr=1e-3, wd=0.0):
    print(f"\n⚠️ INICIANDO TESTE DE ESTRESSE: {model_name} (LR={lr}, WD={wd})")

    # Chamamos a tokenização normal
    tok_train, tok_test, tokenizer_obj = prepare_tokenized_datasets(model_name, df_train, df_test)

    # Criamos os TrainingArguments COM ERROS PROPOSITAIS
    bad_args = TrainingArguments(
        output_dir="./stress_test",
        learning_rate=lr,      # Valor muito alto
        weight_decay=wd,       # Sem regularização
        num_train_epochs=2,    # 2 épocas já bastam para ver o desastre
        eval_strategy="epoch",
        report_to="none"
    )

    # Trainer simplificado para o teste
    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device),
        args=bad_args,
        train_dataset=tok_train.train_test_split(test_size=0.15, seed=42)["train"],
        eval_dataset=tok_train.train_test_split(test_size=0.15, seed=42)["test"],
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer_obj),
        compute_metrics=compute_metrics, # aquela função de F1 que já temos
    )

    trainer.train()
    return trainer.evaluate()

# Rodando o teste
resultado_caos = run_stress_test("distilroberta-base", train_final, test_final)
print(f"📊 F1-Score no Cenário de Estresse: {resultado_caos['eval_f1']:.4f}")


⚠️ INICIANDO TESTE DE ESTRESSE: distilroberta-base (LR=0.001, WD=0.0)
📥 Carregando Tokenizer: distilroberta-base...
✂️ Tokenizando datasets...


Treino:   0%|          | 0/22169 [00:00<?, ? examples/s]

Teste:   0%|          | 0/5712 [00:00<?, ? examples/s]

✅ Tokenização concluída para distilroberta-base
Colunas finais no treino: ['label', 'input_ids', 'attention_mask']


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]